# DRIVE vessel segmentation: simple encoder-decoder CNN (local run)
- Data: the 20 labeled images in `driver/training`, split 12 / 4 / 4 (train / val / test) by image, seed 42
- Labels: `1st_manual` (vessels). The FOV mask (`mask`) only marks the valid retina region; it is used to ignore the black background in the loss and metrics
- Model: encoder-decoder CNN without skip connections
- Output: loss curve, accuracy curve (plus a Dice curve, because accuracy is misleading for thin vessels), and final test metrics

In [ ]:
import os, random, time
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

# ---- config (change here) ----
# DTU-HPC dataset location (DRIVE). If your folder differs, run:  ls /dtu/datasets1/02516  and edit this line.
ROOT = Path("/dtu/datasets1/02516/DRIVE/training")
if not ROOT.exists():
    base = Path("/dtu/datasets1/02516")
    print("ROOT not found. Contents of", base, ":", sorted(os.listdir(base)) if base.exists() else "folder missing")
    raise FileNotFoundError(ROOT)
SIZE = 512                       # images are resized to SIZE x SIZE (try 256 for a faster test)
BATCH_SIZE = 4
EPOCHS = 100
LR = 1e-3
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Split (12 / 4 / 4 by image)

In [ ]:
ids = sorted(int(f.split("_")[0]) for f in os.listdir(ROOT / "images") if f.endswith(".tif"))
assert len(ids) == 20, len(ids)
random.Random(SEED).shuffle(ids)
train_ids, val_ids, test_ids = sorted(ids[:12]), sorted(ids[12:16]), sorted(ids[16:])
print("train:", train_ids); print("val:  ", val_ids); print("test: ", test_ids)

## 2. Load data into memory (resize image bilinear, masks nearest)

In [ ]:
def load_one(i):
    img = Image.open(ROOT / "images" / f"{i}_training.tif").convert("RGB").resize((SIZE, SIZE), Image.BILINEAR)
    lab = Image.open(ROOT / "1st_manual" / f"{i}_manual1.gif").convert("L").resize((SIZE, SIZE), Image.NEAREST)
    fov = Image.open(ROOT / "mask" / f"{i}_training_mask.gif").convert("L").resize((SIZE, SIZE), Image.NEAREST)
    x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.).permute(2, 0, 1)   # 3,H,W
    y = torch.from_numpy((np.asarray(lab) > 127).astype(np.float32))[None]            # 1,H,W  vessels
    m = torch.from_numpy((np.asarray(fov) > 127).astype(np.float32))[None]            # 1,H,W  valid region
    return x, y, m

def load_split(id_list):
    xs, ys, ms = zip(*[load_one(i) for i in id_list])
    return torch.stack(xs), torch.stack(ys), torch.stack(ms)

Xtr, Ytr, Mtr = load_split(train_ids)
Xva, Yva, Mva = load_split(val_ids)
Xte, Yte, Mte = load_split(test_ids)

# normalization: statistics from the TRAIN set only, applied to all splits
mean = Xtr.mean(dim=(0, 2, 3), keepdim=True)
std = Xtr.std(dim=(0, 2, 3), keepdim=True)
Xtr, Xva, Xte = [(x - mean) / std for x in (Xtr, Xva, Xte)]

# class imbalance: vessels are a small fraction of the FOV pixels
pos = (Ytr * Mtr).sum(); neg = ((1 - Ytr) * Mtr).sum()
pos_weight = (neg / pos).to(device)
print("train", Xtr.shape, "| vessel fraction in FOV: %.3f" % (pos / (pos + neg)).item(), "| pos_weight: %.1f" % pos_weight.item())

Quick look at the data (image, vessel label, FOV mask):

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
x0 = Xtr[0] * std[0] + mean[0]
ax[0].imshow(x0.permute(1, 2, 0).clamp(0, 1)); ax[0].set_title("image")
ax[1].imshow(Ytr[0, 0], cmap="gray"); ax[1].set_title("vessels (label)")
ax[2].imshow(Mtr[0, 0], cmap="gray"); ax[2].set_title("FOV mask (not vessels)")
for a in ax: a.axis("off")
plt.show()

## 3. Model: encoder-decoder CNN (3 down, 3 up, no skip connections)

In [ ]:
def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class EncDec(nn.Module):
    def __init__(self, c=(16, 32, 64, 128)):
        super().__init__()
        self.e1, self.e2, self.e3, self.bott = block(3, c[0]), block(c[0], c[1]), block(c[1], c[2]), block(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.u3, self.d3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2), block(c[2], c[2])
        self.u2, self.d2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2), block(c[1], c[1])
        self.u1, self.d1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2), block(c[0], c[0])
        self.out = nn.Conv2d(c[0], 1, 1)           # logits (no sigmoid)

    def forward(self, x):
        x = self.e1(x)                    # H
        x = self.e2(self.pool(x))         # H/2
        x = self.e3(self.pool(x))         # H/4
        x = self.bott(self.pool(x))       # H/8
        x = self.d3(self.u3(x))           # H/4
        x = self.d2(self.u2(x))           # H/2
        x = self.d1(self.u1(x))           # H
        return self.out(x)

model = EncDec().to(device)
print("parameters:", sum(p.numel() for p in model.parameters()))
print("output shape:", model(torch.zeros(1, 3, SIZE, SIZE, device=device)).shape)

## 4. Loss and metrics (both only count pixels inside the FOV)

In [ ]:
def masked_bce(logits, y, m):
    l = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight, reduction="none")
    return (l * m).sum() / m.sum()

def counts(logits, y, m):
    p = (torch.sigmoid(logits) > 0.5).float()
    tp = (p * y * m).sum().item(); fp = (p * (1 - y) * m).sum().item()
    fn = ((1 - p) * y * m).sum().item(); tn = ((1 - p) * (1 - y) * m).sum().item()
    return np.array([tp, fp, fn, tn])

def metrics(c, eps=1e-8):
    tp, fp, fn, tn = c
    return dict(dice=2 * tp / (2 * tp + fp + fn + eps), iou=tp / (tp + fp + fn + eps),
                acc=(tp + tn) / (tp + fp + fn + tn + eps),
                sens=tp / (tp + fn + eps), spec=tn / (tn + fp + eps))

@torch.no_grad()
def evaluate(X, Y, M):
    model.eval(); loss_sum, c = 0.0, np.zeros(4)
    for i in range(0, len(X), BATCH_SIZE):
        x, y, m = X[i:i+BATCH_SIZE].to(device), Y[i:i+BATCH_SIZE].to(device), M[i:i+BATCH_SIZE].to(device)
        lg = model(x)
        loss_sum += masked_bce(lg, y, m).item() * len(x)
        c += counts(lg, y, m)
    return loss_sum / len(X), metrics(c)

## 5. Train (augmentation: random horizontal / vertical flips, training set only)

In [ ]:
def augment(x, y, m):
    if random.random() < 0.5: x, y, m = x.flip(-1), y.flip(-1), m.flip(-1)
    if random.random() < 0.5: x, y, m = x.flip(-2), y.flip(-2), m.flip(-2)
    return x, y, m

opt = torch.optim.Adam(model.parameters(), lr=LR)
hist = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc", "train_dice", "val_dice"]}
best_dice, best_state = -1, None
t0 = time.time()

for ep in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        x, y, m = augment(Xtr[idx], Ytr[idx], Mtr[idx])
        x, y, m = x.to(device), y.to(device), m.to(device)
        opt.zero_grad()
        loss = masked_bce(model(x), y, m)
        loss.backward(); opt.step()

    # epoch metrics (no augmentation, eval mode) so train and val are comparable
    tl, tm = evaluate(Xtr, Ytr, Mtr)
    vl, vm = evaluate(Xva, Yva, Mva)
    for k, v in zip(hist, [tl, vl, tm["acc"], vm["acc"], tm["dice"], vm["dice"]]): hist[k].append(v)
    if vm["dice"] > best_dice:
        best_dice = vm["dice"]; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if ep == 1 or ep % 5 == 0:
        print(f"ep {ep:3d} | loss {tl:.4f}/{vl:.4f} | acc {tm['acc']:.4f}/{vm['acc']:.4f} | dice {tm['dice']:.3f}/{vm['dice']:.3f} | {time.time()-t0:.0f}s")

model.load_state_dict(best_state)
print("best val dice: %.4f (weights restored)" % best_dice)

## 6. Loss curve, accuracy curve (and Dice curve)

In [ ]:
ep_axis = range(1, len(hist["train_loss"]) + 1)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(ep_axis, hist["train_loss"], label="train"); ax[0].plot(ep_axis, hist["val_loss"], label="val")
ax[0].set_title("Loss (weighted BCE)"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep_axis, hist["train_acc"], label="train"); ax[1].plot(ep_axis, hist["val_acc"], label="val")
ax[1].set_title("Accuracy (pixels inside FOV)"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].plot(ep_axis, hist["train_dice"], label="train"); ax[2].plot(ep_axis, hist["val_dice"], label="val")
ax[2].set_title("Dice"); ax[2].set_xlabel("epoch"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig("drive_cnn_curves.png", dpi=150); plt.show()

## 7. Final metrics on train / val / test

In [ ]:
for name, (X, Y, M) in {"train": (Xtr, Ytr, Mtr), "val": (Xva, Yva, Mva), "test": (Xte, Yte, Mte)}.items():
    l, m = evaluate(X, Y, M)
    print(f"{name:5s} loss {l:.4f} | " + " | ".join(f"{k} {v:.4f}" for k, v in m.items()))

## 8. Test predictions

In [ ]:
model.eval()
with torch.no_grad():
    pred = (torch.sigmoid(model(Xte.to(device))) > 0.5).float().cpu()
fig, ax = plt.subplots(len(Xte), 3, figsize=(9, 3 * len(Xte)))
for r in range(len(Xte)):
    img = (Xte[r] * std[0] + mean[0]).permute(1, 2, 0).clamp(0, 1)
    ax[r, 0].imshow(img); ax[r, 1].imshow(Yte[r, 0], cmap="gray"); ax[r, 2].imshow(pred[r, 0] * Mte[r, 0], cmap="gray")
    for a, t in zip(ax[r], [f"image {test_ids[r]}", "ground truth", "prediction"]): a.set_title(t); a.axis("off")
plt.tight_layout(); plt.savefig("drive_cnn_predictions.png", dpi=120); plt.show()